In [2]:
import pandas as pd

print("1. 원본 데이터 불러오는 중...")
# 파일 경로는 강산님의 환경에 맞게 수정해주세요! (예: "TCGA-LUAD.star_fpkm(1).tsv")
file_path = "TCGA-LUAD.star_fpkm (1).tsv" 
df = pd.read_csv(file_path, sep='\t', index_col=0)

print("2. 환자 분류 (암 vs 정상) 및 정답지(Label) 만들기...")
tumor_cols = []
normal_cols = []
labels = {} # 샘플 이름: 정답 (1=암, 0=정상)

for col in df.columns:
    parts = col.split('-')
    if len(parts) >= 4:
        sample_type = parts[3][:2]
        if sample_type.isdigit():
            if int(sample_type) < 10:
                tumor_cols.append(col)
                labels[col] = 1 # 암 (정답 1)
            elif int(sample_type) >= 10:
                normal_cols.append(col)
                labels[col] = 0 # 정상 (정답 0)

print("3. 강산님 & 친구분 마커 (핵심 유전자 30개) 자동 선별 중...")
# 먼저 평균과 LogFC를 계산합니다.
df['LogFC'] = df[tumor_cols].mean(axis=1) - df[normal_cols].mean(axis=1)

# 강산님 마커: 가장 많이 폭증한 유전자 Top 15
top_genes = df.sort_values(by='LogFC', ascending=False).head(15).index.tolist()
# 친구분 마커: 가장 많이 파괴된 유전자 Bottom 15
bottom_genes = df.sort_values(by='LogFC', ascending=True).head(15).index.tolist()

# 30개 드림팀 결성! (이제 전체 6만 개 유전자 중 이 30개만 남깁니다)
selected_features = top_genes + bottom_genes

print("4. AI가 공부하기 좋은 형태로 표 뒤집기(Transpose)...")
# 1) 30개 핵심 유전자 데이터만 싹둑 잘라냅니다.
ml_data = df.loc[selected_features]

# 2) 표를 뒤집습니다. (T = Transpose) 
# 이제 행(가로)이 '환자'가 되고, 열(세로)이 '유전자'가 됩니다!
ml_data = ml_data.T

# 3) 맨 끝에 이 환자가 암인지 아닌지 '정답(Label)' 컬럼을 붙여줍니다.
ml_data['Label'] = ml_data.index.map(labels)

# 4) 혹시 모를 에러를 막기 위해 분류가 안 된 샘플은 삭제하고, 정답을 정수(0, 1)로 바꿉니다.
ml_data.dropna(subset=['Label'], inplace=True)
ml_data['Label'] = ml_data['Label'].astype(int)

print("\n==============================================================")
print("📊 [미리보기] 완성된 AI 학습용 데이터셋 (처음 5명)")
print("==============================================================")
print(ml_data.head())
print(f"\n데이터 크기: 총 환자 {ml_data.shape[0]}명, 시험과목(유전자) {ml_data.shape[1]-1}개")

# 엑셀(CSV)로 저장!
ml_data.to_csv("ML_Ready_Dataset.csv")
print("\n✅ Step 1 완료! 'ML_Ready_Dataset.csv' 파일이 성공적으로 저장되었습니다.")

1. 원본 데이터 불러오는 중...
2. 환자 분류 (암 vs 정상) 및 정답지(Label) 만들기...
3. 강산님 & 친구분 마커 (핵심 유전자 30개) 자동 선별 중...


/var/folders/6g/7fmgk_ln1979hm_m4d8kd8g40000gn/T/ipykernel_2168/2035425001.py:27: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['LogFC'] = df[tumor_cols].mean(axis=1) - df[normal_cols].mean(axis=1)


4. AI가 공부하기 좋은 형태로 표 뒤집기(Transpose)...

📊 [미리보기] 완성된 AI 학습용 데이터셋 (처음 5명)
Ensembl_ID        ENSG00000118785.14  ENSG00000143320.9  ENSG00000147689.16  \
TCGA-38-7271-01A            5.444992           8.685850            3.670002   
TCGA-55-7914-01A            8.257108           2.153676            3.035554   
TCGA-95-7043-01A            8.446405           3.719282            4.752914   
TCGA-73-4658-01A            9.685289           7.946273            5.054041   
TCGA-86-8076-01A            6.129637           4.659371            4.081340   

Ensembl_ID        ENSG00000164266.11  ENSG00000105388.16  ENSG00000164932.13  \
TCGA-38-7271-01A            4.905187            7.580918            3.875603   
TCGA-55-7914-01A            0.823831            0.742782            4.824651   
TCGA-95-7043-01A            2.324407           10.777998            3.977124   
TCGA-73-4658-01A            5.160815            3.668618            6.590368   
TCGA-86-8076-01A            4.411847            2.18